#  LaLiga Match Outcome Prediction Model — Random Forest

In this notebook, we convert data types of a few columns, make some new categorical columns and make rolling averages for various columns.

And then, we build and evaluate a Random Forest model to predict match outcomes (Win/Draw/Loss) based on team statistics.

Goal: Predict La Liga match outcomes (Win/Draw/Loss) using team statistics

Approach:
- Features: Rolling 3-game averages of goals, xG, possession, and many such team statistics + contextual factors like the opponent, hour of the day, etc.
- Algorithm: Random Forest
- Train/Test split: Temporal (pre-2025 / 2025+) to simulate real-world prediction
- Evaluation: Multi-class accuracy, precision, recall per outcome

In [19]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, classification_report

In [20]:
# Load the data

matches = pd.read_csv('merged.csv')

matches.head()

,league,season,team,game,date,time,round,day,venue,result,...,long.1_att,long.2_cmp%,ast,xag,xa,kp,1/3,ppa,crspa,prgp
0,ESP-La Liga,2122,Alavés,2021-08-14 Alavés-Real Madrid,2021-08-14,22:00:00,Matchweek 1,Sat,Home,L,...,69,75.4,0.0,0.7,0.3,7.0,19.0,6.0,5.0,20.0
1,ESP-La Liga,2122,Alavés,2021-08-21 Alavés-Mallorca,2021-08-21,17:00:00,Matchweek 2,Sat,Home,L,...,87,46.0,0.0,0.3,0.5,6.0,16.0,4.0,1.0,21.0
2,ESP-La Liga,2122,Alavés,2021-08-27 Valencia-Alavés,2021-08-27,22:15:00,Matchweek 3,Fri,Away,L,...,84,60.7,0.0,0.4,0.3,8.0,22.0,4.0,1.0,24.0
3,ESP-La Liga,2122,Alavés,2021-09-18 Alavés-Osasuna,2021-09-18,21:00:00,Matchweek 5,Sat,Home,L,...,93,38.7,0.0,0.1,0.8,4.0,25.0,2.0,0.0,28.0
4,ESP-La Liga,2122,Alavés,2021-09-22 Espanyol-Alavés,2021-09-22,19:30:00,Matchweek 6,Wed,Away,L,...,86,58.1,0.0,0.4,0.6,5.0,38.0,11.0,5.0,39.0


In [21]:
# Convert date to datetime

matches['date'] = pd.to_datetime(matches['date'])

# Creat target variable (Win=2, Draw=1, Loss=0)

matches['target'] = matches['result'].map({'W': 2, 'D': 1, 'L': 0})


# Encode categorical variables

matches['venue_code'] = matches['venue'].astype('category').cat.codes
matches['opp_code'] = matches['opponent'].astype('category').cat.codes
matches['hour'] = matches['time'].str.replace(":.+", "", regex=True).astype('int')
matches['day_code'] = matches['date'].dt.dayofweek


In [22]:
# Select important statistics to track

stats_cols = [
    'gf', 'ga', 'xg', 'xga', 'poss',
    'touches_touches', 'touches.1_def pen', 'touches.2_def 3rd', 
    'touches.3_mid 3rd', 'touches.4_att 3rd', 'touches.5_att pen', 
    'touches.6_live', 'take-ons_att', 'take-ons.1_succ', 'take-ons.2_succ%', 
    'take-ons.3_tkld', 'take-ons.4_tkld%', 'carries_carries', 
    'carries.1_totdist', 'carries.2_prgdist', 'carries.3_prgc', 
    'carries.4_1/3', 'carries.5_cpa', 'carries.6_mis', 'carries.7_dis',
    'receiving_rec', 'receiving.1_prgr', 'standard_gls', 'standard.1_sh', 
    'standard.2_sot', 'standard.7_fk', 'standard.8_pk', 'standard.9_pkatt',
    'expected_xg', 'expected.1_npxg', 'expected.3_g-xg', 'expected.4_np:g-xg',
    'sca types_sca', 'sca types.1_passlive', 'sca types.2_passdead', 
    'sca types.3_to', 'sca types.4_sh', 'sca types.5_fld', 'sca types.6_def',
    'gca types_gca', 'gca types.1_passlive', 'gca types.2_passdead', 
    'gca types.3_to', 'gca types.4_sh', 'gca types.5_fld', 'gca types.6_def',
    'total_cmp', 'total.1_att', 'total.2_cmp%', 'total.3_totdist', 
    'total.4_prgdist', 'short_cmp', 'short.1_att', 'short.2_cmp%',
    'medium_cmp', 'medium.1_att', 'medium.2_cmp%', 'long_cmp', 'long.1_att',
    'long.2_cmp%', 'ast', 'xag', 'xa', 'kp', '1/3', 'ppa', 'crspa', 'prgp'
]

# Defining a rolling averages function
# Rolling averages capture recent team form without data leakage

def calculate_rolling_averages(group, cols):
    """Calculate 3-game rolling average for each team"""
    group = group.sort_values('date')
    rolling_stats = group[cols].rolling(window=3, closed='left').mean()

    # Add '_rolling' suffix to column names
    new_cols = [f'{c}_rolling' for c in cols]
    group[new_cols] = rolling_stats

    # Remove rows with NaN (first 3 games per team)
    group = group.dropna(subset=new_cols)
    return group

In [23]:
# Apply rolling averages per team

matches_rolling = matches.groupby('team').apply(
    lambda x: calculate_rolling_averages(x, stats_cols)
)
matches_rolling = matches_rolling.reset_index(drop=True)



In [24]:
# Split by date

train = matches_rolling[matches_rolling['date'] < '2025-01-01']
test = matches_rolling[matches_rolling['date'] >= '2025-01-01']

In [25]:
# Define the features to use

basic_features = ['venue_code', 'opp_code', 'hour', 'day_code']
rolling_features = [f'{c}_rolling' for c in stats_cols]
all_features = basic_features + rolling_features

In [26]:
# Initialize the Random Forest

rf = RandomForestClassifier(
    n_estimators=100,        # Number of trees
    max_depth=15,            # Maximum depth of each tree
    min_samples_split=10,    # Minimum samples to split a node
    random_state=42,         # For reproducibility
    n_jobs=-1                # Use all CPU cores
)

In [27]:
rf.fit(train[all_features], train['target'])

RandomForestClassifier(max_depth=15, min_samples_split=10, n_jobs=-1,
                       random_state=42)

In [28]:
# Predict on the test set
predictions = rf.predict(test[all_features])

# Calculate the accuracy
accuracy = accuracy_score(test['target'], predictions)
print(accuracy)

0.47738693467336685


In [29]:
# Calculate the precision

precision = precision_score(test['target'], predictions, average=None)

print(precision)

[0.45812808 0.35       0.51428571]


In [30]:
print(classification_report(test['target'], predictions,
                           target_names=['Loss', 'Draw', 'Win']))

              precision    recall  f1-score   support

        Loss       0.46      0.62      0.53       150
        Draw       0.35      0.07      0.12        98
         Win       0.51      0.60      0.55       150

    accuracy                           0.48       398
   macro avg       0.44      0.43      0.40       398
weighted avg       0.45      0.48      0.44       398



In [31]:
# Create the results dataframe

results = pd.DataFrame({
    'actual': test['target'],
    'predicted': predictions,
    'correct': test['target'] == predictions
})


In [32]:
results

,actual,predicted,correct
91,0,0,True
92,2,0,False
93,1,1,True
94,0,0,True
95,0,2,False
...,...,...,...
2960,2,2,True
2961,2,0,False
2962,2,0,False
2963,2,0,False


In [33]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': all_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))


Top 10 Most Important Features:
                      feature  importance
0                  venue_code    0.022214
1                    opp_code    0.022167
68        long.2_cmp%_rolling    0.021252
57       total.2_cmp%_rolling    0.019267
60          short_cmp_rolling    0.019198
22  carries.1_totdist_rolling    0.019069
61        short.1_att_rolling    0.018692
13  touches.4_att 3rd_rolling    0.018507
62       short.2_cmp%_rolling    0.018463
20   take-ons.4_tkld%_rolling    0.018427


In [34]:
# Class-specific analysis
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test['target'], predictions)

print("\nDetailed performance by class:")
print("Confusion Matrix:")
print(pd.DataFrame(cm,
                   index=['Actual Loss', 'Actual Draw', 'Actual Win'],
                   columns=['Pred Loss', 'Pred Draw', 'Pred Win']))


Detailed performance by class:
Confusion Matrix:
             Pred Loss  Pred Draw  Pred Win
Actual Loss         93          9        48
Actual Draw         54          7        37
Actual Win          56          4        90


In [35]:
# Baseline comparison
from collections import Counter
most_common = Counter(train['target']).most_common(1)[0][0]
baseline_preds = [most_common] * len(test)
baseline_acc = accuracy_score(test['target'], baseline_preds)

print(f"\n{'='*50}")
print(f"BASELINE vs OUR MODEL")
print(f"{'='*50}")
print(f"Baseline (always predict most common class): {baseline_acc:.1%}")
print(f"Our Random Forest Model: {accuracy:.1%}")
print(f"Improvement: {(accuracy - baseline_acc) / baseline_acc * 100:+.1f}%")
print(f"{'='*50}")


BASELINE vs OUR MODEL
Baseline (always predict most common class): 37.7%
Our Random Forest Model: 47.7%
Improvement: +26.7%


## Model Limitations and Future Work

MODEL LIMITATIONS:
1. Small test set (~400 matches, only 2025 data)
2. Class imbalance: Draws are much harder to predict than Wins/Losses
3. No hyperparameter tuning beyond basic grid
4. Does not account for tactical adjustments or managerial changes

POTENTIAL IMPROVEMENTS:
1. Address class imbalance using SMOTE or class weights
3. Grid search for optimal hyperparameters
4. Add opponent strength metrics (Elo ratings)
5. Incorporate transfer market data
6. Add head-to-head historical records
7. Expand to multiple leagues for better generalization
